# Notebook 1 — Acquisition et constitution du dataset openSenseMap

## Position dans le pipeline

`NB1 (acquisition) → NB1bis viz (optionnel) → NB2 (QC) → NB3 (confiance) → NB4 (mémoire)`

Guide détaillé : `00_PIPELINE_PAS_A_PAS.md`

## Objectif

Sélectionner des stations openSenseMap **réellement actives**, récupérer température, humidité et pression sur ~30 jours, produire un **dataset brut au format long**.

## Sorties attendues

| Fichier | Rôle |
|---|---|
| `data/metadata/stations_selectionnees_actives.csv` | stations retenues |
| `data/raw/opensensemap_raw_long_active_50.csv` | mesures brutes (source QC) |
| `data/raw/download_checkpoint.parquet` | reprise en cas d'interruption |
| `data/metadata/download_log_active_50.csv` | journal des requêtes |
| `data/metadata/error_log_active_50.csv` | erreurs réseau/API |

## Format des mesures

Une ligne = une observation. Colonnes clés : `station_id`, `phenomenon` (`temperature` / `humidity` / `pressure`), `timestamp`, `value`, `sensor_unit`.

> Les colonnes `temperature_last_value` du fichier stations sont des **métadonnées** (parfois incomplètes). Les vraies valeurs numériques sont dans le CSV brut (`value`).

## Si le dataset brut existe déjà

Inutile de retélécharger. Vérifier le fichier puis passer au **Notebook 2** (ou au notebook de visualisation lisible).

Le téléchargement ci-dessous reprend la logique robuste de `download_opensensemap_thp.py` (retries, backoff, checkpoint parquet). Alternative CLI :

```bash
python download_opensensemap_thp.py
```


## 0. Contrôle rapide — dataset déjà disponible ?

Si `data/raw/opensensemap_raw_long_active_50.csv` existe déjà (téléchargement antérieur), vous pouvez **sauter** les cellules de téléchargement et passer directement au Notebook 2.

Si un téléchargement a été interrompu, relancer la cellule de téléchargement : elle reprend automatiquement depuis `data/raw/download_checkpoint.parquet`.

Pour enchaîner tout le pipeline : ouvrir `Notebook_0_Executer_pipeline_pas_a_pas.ipynb`.


In [ ]:
import requests
import pandas as pd
import numpy as np
import time
from datetime import datetime, timezone, timedelta
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

print("Bibliothèques importées avec succès.")


In [ ]:
BASE_URL = "https://api.opensensemap.org"

# Allemagne approximative : lon_min, lat_min, lon_max, lat_max
BBOX = "5.5,47.0,15.5,55.1"

# Période de 30 jours (données vérifiées disponibles sur l'API)
FROM_DATE = "2026-06-15T00:00:00Z"
TO_DATE = "2026-07-15T00:00:00Z"

EXPOSURE = "outdoor"
TARGET_STATIONS = 50
MAX_BOXES_TO_SCAN = 1000
MIN_MEASUREMENTS_PER_SENSOR = 100
CHUNK_DAYS = 7  # évite le plafond API (~10 000 points / requête)
FETCH_RETRIES = 5
FETCH_TIMEOUT = 120
SLEEP_BETWEEN_REQUESTS = 0.4
CHECKPOINT_EVERY = 10

# Périmètre du mémoire : température, humidité, pression uniquement.
# Les senseBox peuvent aussi exposer luminosité / PM, hors scope ici.
TARGET_PHENOMENA = ["temperature", "humidity", "pressure"]

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_DIR = DATA_DIR / "metadata"

for folder in [DATA_DIR, RAW_DIR, PROCESSED_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = RAW_DIR / "download_checkpoint.parquet"
RAW_LONG_PATH = RAW_DIR / "opensensemap_raw_long_active_50.csv"
DOWNLOAD_LOG_PATH = METADATA_DIR / "download_log_active_50.csv"
ERROR_LOG_PATH = METADATA_DIR / "error_log_active_50.csv"
STATIONS_METADATA_PATH = METADATA_DIR / "stations_selectionnees_actives.csv"

print("Paramètres définis.")
print("BBOX :", BBOX)
print("Période :", FROM_DATE, "→", TO_DATE)
print("Stations cibles :", TARGET_STATIONS)
print("Checkpoint :", CHECKPOINT_PATH)


In [ ]:
TEMPERATURE_NAMES = [
    "temperature", "temperatur", "temp", "air temperature",
    "lufttemperatur", "bme280 temperature", "bme680 temperature"
]

HUMIDITY_NAMES = [
    "humidity", "luftfeuchte", "relative humidity", "rel. humidity",
    "air humidity", "feuchtigkeit", "bme280 humidity", "bme680 humidity"
]

PRESSURE_NAMES = [
    "pressure", "luftdruck", "air pressure", "atmospheric pressure",
    "barometric pressure", "druck", "bme280 pressure", "bme680 pressure"
]

print("Variables recherchées : température, humidité, pression.")


In [ ]:
def normalize_text(text):
    if text is None:
        return ""
    return str(text).strip().lower()

def clean_unit(unit):
    if pd.isna(unit):
        return ""
    return str(unit).strip().lower()

def extract_coordinates(box):
    location = box.get("currentLocation", {})
    coordinates = location.get("coordinates", [None, None])

    if not coordinates or len(coordinates) < 2:
        return None, None

    lon = coordinates[0]
    lat = coordinates[1]
    return lat, lon

def sensor_matches(sensor, possible_names):
    possible_names = [normalize_text(name) for name in possible_names]

    title = normalize_text(sensor.get("title"))
    unit = normalize_text(sensor.get("unit"))
    sensor_type = normalize_text(sensor.get("sensorType"))

    combined_text = f"{title} {unit} {sensor_type}"

    return any(name in combined_text for name in possible_names)

def find_sensor_by_names(sensors, possible_names):
    for sensor in sensors:
        if sensor_matches(sensor, possible_names):
            return sensor
    return None

def get_last_measurement_info(sensor):
    """Extrait la dernière mesure si l'API renvoie un objet {value, createdAt}.

    Si lastMeasurement est seulement un identifiant (str), on n'en fait pas
    une valeur numérique : cela évite de polluer temperature_last_value, etc.
    """
    last_measurement = sensor.get("lastMeasurement", None)

    if isinstance(last_measurement, dict):
        return {
            "last_value": last_measurement.get("value"),
            "last_created_at": last_measurement.get("createdAt"),
        }

    # Identifiant seul ou format inattendu → pas de valeur exploitable
    return {
        "last_value": None,
        "last_created_at": None,
    }

print("Fonctions utilitaires définies.")


In [ ]:
import requests
import time

def get_boxes(
    bbox,
    exposure="outdoor",
    max_boxes=1000,
    base_url=BASE_URL,
    timeout=120,
    retries=5,
    sleep_between_retries=5
):
    """
    Récupère les stations openSenseMap situées dans une zone donnée.

    Cette version est robuste contre les coupures réseau temporaires :
    - ChunkedEncodingError
    - IncompleteRead
    - Timeout
    - erreurs temporaires de connexion
    """

    url = f"{base_url}/boxes"

    params = {
        "bbox": bbox,
        "exposure": exposure
    }

    headers = {
        "User-Agent": "Mozilla/5.0 - academic data quality study"
    }

    last_error = None

    for attempt in range(1, retries + 1):
        print(f"Tentative {attempt}/{retries}")
        print("Requête envoyée à l'API openSenseMap...")
        print("URL :", url)
        print("Paramètres :", params)

        try:
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=timeout
            )

            response.raise_for_status()

            boxes = response.json()

            if not isinstance(boxes, list):
                raise TypeError(
                    "La réponse de l'API n'est pas une liste de stations."
                )

            print(f"Nombre de stations récupérées avant limitation : {len(boxes)}")

            boxes = boxes[:max_boxes]

            print(f"Nombre de stations conservées pour inspection : {len(boxes)}")

            return boxes

        except requests.exceptions.ChunkedEncodingError as e:
            last_error = e
            print("Connexion interrompue pendant la lecture de la réponse.")
            print("Nouvelle tentative après pause...")

        except requests.exceptions.ReadTimeout as e:
            last_error = e
            print("Temps d'attente dépassé.")
            print("Nouvelle tentative après pause...")

        except requests.exceptions.ConnectionError as e:
            last_error = e
            print("Erreur de connexion.")
            print("Nouvelle tentative après pause...")

        except Exception as e:
            last_error = e
            print("Erreur inattendue :", repr(e))
            print("Nouvelle tentative après pause...")

        time.sleep(sleep_between_retries)

    raise RuntimeError(
        "Échec de récupération des stations après plusieurs tentatives. "
        f"Dernière erreur : {repr(last_error)}"
    )

boxes = get_boxes(
    bbox=BBOX,
    exposure=EXPOSURE,
    max_boxes=MAX_BOXES_TO_SCAN
)

print("Récupération terminée.")
print("Nombre de stations inspectables :", len(boxes))

if len(boxes) > 0:
    example_box = boxes[0]
    print("\nExemple d'une station récupérée :")
    print("ID :", example_box.get("_id"))
    print("Nom :", example_box.get("name"))
    print("Exposition :", example_box.get("exposure"))
    print("Nombre de capteurs :", len(example_box.get("sensors", [])))
else:
    print("Aucune station trouvée.")


In [ ]:
selected_stations = []

for box in boxes:
    box_id = box.get("_id")
    name = box.get("name")
    exposure = box.get("exposure")
    sensors = box.get("sensors", [])

    lat, lon = extract_coordinates(box)

    if lat is None or lon is None:
        continue

    temp_sensor = find_sensor_by_names(sensors, TEMPERATURE_NAMES)
    hum_sensor = find_sensor_by_names(sensors, HUMIDITY_NAMES)
    pres_sensor = find_sensor_by_names(sensors, PRESSURE_NAMES)

    if temp_sensor and hum_sensor and pres_sensor:
        temp_last = get_last_measurement_info(temp_sensor)
        hum_last = get_last_measurement_info(hum_sensor)
        pres_last = get_last_measurement_info(pres_sensor)

        selected_stations.append({
            "box_id": box_id,
            "station_name": name,
            "exposure": exposure,
            "latitude": lat,
            "longitude": lon,

            "temperature_sensor_id": temp_sensor.get("_id"),
            "temperature_title": temp_sensor.get("title"),
            "temperature_unit": temp_sensor.get("unit"),
            "temperature_sensor_type": temp_sensor.get("sensorType"),
            "temperature_last_value": temp_last["last_value"],
            "temperature_last_created_at": temp_last["last_created_at"],

            "humidity_sensor_id": hum_sensor.get("_id"),
            "humidity_title": hum_sensor.get("title"),
            "humidity_unit": hum_sensor.get("unit"),
            "humidity_sensor_type": hum_sensor.get("sensorType"),
            "humidity_last_value": hum_last["last_value"],
            "humidity_last_created_at": hum_last["last_created_at"],

            "pressure_sensor_id": pres_sensor.get("_id"),
            "pressure_title": pres_sensor.get("title"),
            "pressure_unit": pres_sensor.get("unit"),
            "pressure_sensor_type": pres_sensor.get("sensorType"),
            "pressure_last_value": pres_last["last_value"],
            "pressure_last_created_at": pres_last["last_created_at"],
        })

selected_stations_df = pd.DataFrame(selected_stations)

print("Stations avec température + humidité + pression :", len(selected_stations_df))

display(selected_stations_df.head())


In [ ]:
selected_stations_clean_df = selected_stations_df.copy()

selected_stations_clean_df["temperature_unit_clean"] = selected_stations_clean_df["temperature_unit"].apply(clean_unit)
selected_stations_clean_df["humidity_unit_clean"] = selected_stations_clean_df["humidity_unit"].apply(clean_unit)
selected_stations_clean_df["pressure_unit_clean"] = selected_stations_clean_df["pressure_unit"].apply(clean_unit)

valid_temperature_units = ["°c", "c", "c°", "° c", "celsius", "ºc"]
valid_humidity_units = ["%", "%rf", "rel%"]
valid_pressure_units = ["hpa", "pa", "pascal", "millibar", "mbar"]

selected_stations_clean_df = selected_stations_clean_df[
    selected_stations_clean_df["temperature_unit_clean"].isin(valid_temperature_units)
    & selected_stations_clean_df["humidity_unit_clean"].isin(valid_humidity_units)
    & selected_stations_clean_df["pressure_unit_clean"].isin(valid_pressure_units)
].copy()

print("Stations avant nettoyage :", len(selected_stations_df))
print("Stations après nettoyage :", len(selected_stations_clean_df))

print("\nUnités de pression après nettoyage :")
print(selected_stations_clean_df["pressure_unit"].value_counts(dropna=False))


In [ ]:
def quick_count_sensor_data(box_id, sensor_id, from_date, to_date):
    url = f"{BASE_URL}/boxes/{box_id}/data/{sensor_id}"
    params = {"from-date": from_date, "to-date": to_date}

    try:
        response = requests.get(url, params=params, timeout=30)

        if response.status_code != 200:
            return 0

        data = response.json()

        if isinstance(data, list):
            return len(data)

        if isinstance(data, dict):
            return 1

        return 0

    except Exception:
        return 0

active_stations = []

print("Recherche de stations réellement actives...")
print("Période :", FROM_DATE, "→", TO_DATE)
print("Stations candidates :", len(selected_stations_clean_df))
print("-" * 80)

for idx, station in selected_stations_clean_df.iterrows():
    box_id = station["box_id"]
    station_name = station["station_name"]

    temp_count = quick_count_sensor_data(box_id, station["temperature_sensor_id"], FROM_DATE, TO_DATE)
    hum_count = quick_count_sensor_data(box_id, station["humidity_sensor_id"], FROM_DATE, TO_DATE)
    pres_count = quick_count_sensor_data(box_id, station["pressure_sensor_id"], FROM_DATE, TO_DATE)

    print(f"{idx+1}/{len(selected_stations_clean_df)} | {station_name} | T={temp_count}, H={hum_count}, P={pres_count}")

    if (
        temp_count >= MIN_MEASUREMENTS_PER_SENSOR
        and hum_count >= MIN_MEASUREMENTS_PER_SENSOR
        and pres_count >= MIN_MEASUREMENTS_PER_SENSOR
    ):
        station_copy = station.copy()
        station_copy["temperature_n_available"] = temp_count
        station_copy["humidity_n_available"] = hum_count
        station_copy["pressure_n_available"] = pres_count
        active_stations.append(station_copy)

    if len(active_stations) >= TARGET_STATIONS:
        break

active_stations_df = pd.DataFrame(active_stations)

print("-" * 80)
print("Stations actives trouvées :", len(active_stations_df))

display(active_stations_df[
    ["box_id", "station_name", "temperature_n_available", "humidity_n_available", "pressure_n_available"]
].head())


In [ ]:
if active_stations_df.empty:
    raise ValueError(
        "Aucune station active trouvée. "
        "Change la période ou diminue MIN_MEASUREMENTS_PER_SENSOR."
    )

stations_df = active_stations_df.head(TARGET_STATIONS).copy().reset_index(drop=True)

stations_df.to_csv(STATIONS_METADATA_PATH, index=False, encoding="utf-8")

print("Stations actives retenues :", len(stations_df))
print("Fichier sauvegardé :", STATIONS_METADATA_PATH)

display(stations_df[
    [
        "box_id", "station_name", "exposure", "latitude", "longitude",
        "temperature_n_available", "humidity_n_available", "pressure_n_available",
    ]
].head())


In [ ]:
def fetch_sensor_chunk(
    box_id,
    sensor_id,
    from_date,
    to_date,
    base_url=BASE_URL,
    timeout=FETCH_TIMEOUT,
    retries=FETCH_RETRIES,
):
    """GET /boxes/{id}/data/{sensor} avec retries + backoff exponentiel.

    Aligné sur download_opensensemap_thp.py :
    - timeout long (réseau instable)
    - retry sur 429 / 5xx et exceptions réseau
    - abandon propre → DataFrame vide
    """
    url = f"{base_url}/boxes/{box_id}/data/{sensor_id}"
    params = {"from-date": from_date, "to-date": to_date}

    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=timeout)

            if response.status_code == 200:
                data = response.json()
                if isinstance(data, list):
                    return pd.DataFrame(data)
                if data:
                    return pd.DataFrame([data])
                return pd.DataFrame()

            # Erreurs non retriables
            if response.status_code not in {429, 500, 502, 503, 504}:
                return pd.DataFrame()

        except requests.RequestException:
            time.sleep(min(30, 2 ** attempt))
            continue

        time.sleep(min(30, 2 ** attempt))

    return pd.DataFrame()

print("Fonction fetch_sensor_chunk prête (retries + backoff).")


In [ ]:
def generate_time_chunks(from_date, to_date, chunk_days=7):
    """Découpe [from_date, to_date] en fenêtres de chunk_days jours (plafond API)."""
    start = datetime.strptime(from_date, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
    end = datetime.strptime(to_date, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)

    chunks = []
    current = start
    while current < end:
        nxt = min(current + timedelta(days=chunk_days), end)
        chunks.append((
            current.strftime("%Y-%m-%dT%H:%M:%SZ"),
            nxt.strftime("%Y-%m-%dT%H:%M:%SZ"),
        ))
        current = nxt
    return chunks

SENSOR_COLUMNS = {
    "temperature": "temperature_sensor_id",
    "humidity": "humidity_sensor_id",
    "pressure": "pressure_sensor_id",
}

time_chunks = generate_time_chunks(FROM_DATE, TO_DATE, chunk_days=CHUNK_DAYS)

print("Nombre de tranches :", len(time_chunks))
for i, (start, end) in enumerate(time_chunks, start=1):
    print(f"Tranche {i} : {start} → {end}")


In [ ]:
frames = []
done = set()
download_log = []
error_log = []

if CHECKPOINT_PATH.exists():
    ck = pd.read_parquet(CHECKPOINT_PATH)
    frames.append(ck)
    done = set(zip(ck["station_id"], ck["phenomenon"], ck["chunk_start"], ck["chunk_end"]))
    print(f"Reprise depuis checkpoint : {len(ck):,} lignes, {len(done)} tâches déjà faites")
else:
    print("Aucun checkpoint — démarrage à zéro.")

tasks = [
    (station, phenomenon, sensor_col, chunk_start, chunk_end)
    for _, station in stations_df.iterrows()
    for phenomenon, sensor_col in SENSOR_COLUMNS.items()
    for chunk_start, chunk_end in time_chunks
    if (station["box_id"], phenomenon, chunk_start, chunk_end) not in done
]

print(f"Stations : {len(stations_df)}")
print(f"Tâches restantes : {len(tasks)}")
print("-" * 80)

for i, (station, phenomenon, sensor_col, chunk_start, chunk_end) in enumerate(tasks, 1):
    box_id = station["box_id"]
    station_name = station["station_name"]
    sensor_id = station[sensor_col]

    print(
        f"[{i}/{len(tasks)}] {station_name} | {phenomenon} | {chunk_start[:10]}",
        flush=True,
    )

    df = fetch_sensor_chunk(box_id, sensor_id, chunk_start, chunk_end)

    log_row = {
        "station_id": box_id,
        "station_name": station_name,
        "phenomenon": phenomenon,
        "sensor_id": sensor_id,
        "chunk_start": chunk_start,
        "chunk_end": chunk_end,
        "n_rows": len(df),
    }
    download_log.append(log_row)

    if df.empty:
        error_log.append({**log_row, "error": "empty_or_failed"})
        print("  vide/erreur", flush=True)
        time.sleep(SLEEP_BETWEEN_REQUESTS)
        continue

    df = df.assign(
        station_id=box_id,
        station_name=station_name,
        latitude=station["latitude"],
        longitude=station["longitude"],
        exposure=station["exposure"],
        phenomenon=phenomenon,
        sensor_id=sensor_id,
        sensor_title=station.get(f"{phenomenon}_title"),
        sensor_unit=station.get(f"{phenomenon}_unit"),
        sensor_type=station.get(f"{phenomenon}_sensor_type"),
        chunk_start=chunk_start,
        chunk_end=chunk_end,
    )
    frames.append(df)
    print(f"  +{len(df)}", flush=True)

    if i % CHECKPOINT_EVERY == 0 or i == len(tasks):
        pd.concat(frames, ignore_index=True).to_parquet(CHECKPOINT_PATH, index=False)
        print(f"  checkpoint → {CHECKPOINT_PATH.name}", flush=True)

    time.sleep(SLEEP_BETWEEN_REQUESTS)

print("-" * 80)
print("Téléchargement terminé.")
print("Blocs en mémoire :", len(frames))
print("Erreurs / vides :", len(error_log))


In [ ]:
if not frames:
    raise RuntimeError(
        "Aucune donnée récupérée. Vérifier le réseau / l'API, "
        "ou relancer après reprise du checkpoint."
    )

raw_df = pd.concat(frames, ignore_index=True)

print("Dataset brut construit.")
print("Nombre total de lignes :", len(raw_df))
print("Nombre total de colonnes :", raw_df.shape[1])
print("Colonnes :", raw_df.columns.tolist())

display(raw_df.head())


In [ ]:
raw_long_df = raw_df.copy()

time_column = next(
    (c for c in ["createdAt", "timestamp", "time", "created_at"] if c in raw_long_df.columns),
    None,
)
if time_column is None:
    raise ValueError(
        f"Aucune colonne temporelle reconnue. Colonnes : {raw_long_df.columns.tolist()}"
    )
if "value" not in raw_long_df.columns:
    raise ValueError(
        f"Aucune colonne value trouvée. Colonnes : {raw_long_df.columns.tolist()}"
    )

raw_long_df["timestamp"] = pd.to_datetime(raw_long_df[time_column], utc=True, errors="coerce")
raw_long_df["value"] = pd.to_numeric(raw_long_df["value"], errors="coerce")

columns_to_keep = [
    "station_id", "station_name", "latitude", "longitude", "exposure",
    "phenomenon", "sensor_id", "sensor_title", "sensor_unit", "sensor_type",
    "timestamp", "value", "chunk_start", "chunk_end",
]

raw_long_df = (
    raw_long_df[columns_to_keep]
    .drop_duplicates(["station_id", "phenomenon", "timestamp", "value"])
    .sort_values(["station_id", "phenomenon", "timestamp"])
    .reset_index(drop=True)
)

raw_long_df.to_csv(RAW_LONG_PATH, index=False, encoding="utf-8")
pd.DataFrame(download_log).to_csv(DOWNLOAD_LOG_PATH, index=False, encoding="utf-8")
pd.DataFrame(error_log).to_csv(ERROR_LOG_PATH, index=False, encoding="utf-8")

print(f"OK → {RAW_LONG_PATH} ({len(raw_long_df):,} lignes, {raw_long_df['station_id'].nunique()} stations)")
print(f"Logs → {DOWNLOAD_LOG_PATH.name} | erreurs → {ERROR_LOG_PATH.name} ({len(error_log)})")
if CHECKPOINT_PATH.exists():
    print(f"Checkpoint conservé : {CHECKPOINT_PATH} (supprimable une fois le CSV validé)")

display(raw_long_df.head())


In [ ]:
print("VÉRIFICATION FINALE — NOTEBOOK 1")
print("=" * 80)

print("Nombre total de mesures :", len(raw_long_df))
print("Nombre de stations :", raw_long_df["station_id"].nunique())
print("Nombre de variables :", raw_long_df["phenomenon"].nunique())

print("\nPériode réelle observée :")
print("Début :", raw_long_df["timestamp"].min())
print("Fin   :", raw_long_df["timestamp"].max())

print("\nNombre de mesures par variable :")
display(raw_long_df["phenomenon"].value_counts())

print("\nNombre de mesures par station — top 10 :")
measures_by_station = (
    raw_long_df
    .groupby(["station_id", "station_name"])
    .size()
    .reset_index(name="n_measurements")
    .sort_values("n_measurements", ascending=False)
)
display(measures_by_station.head(10))

print("\nNombre de mesures par station et par variable — aperçu :")
measures_by_station_variable = (
    raw_long_df
    .groupby(["station_id", "station_name", "phenomenon"])
    .size()
    .reset_index(name="n_measurements")
    .sort_values(["station_name", "phenomenon"])
)
display(measures_by_station_variable.head(20))

print("\nValeurs manquantes dans les colonnes essentielles :")
display(raw_long_df[["station_id", "phenomenon", "timestamp", "value"]].isna().sum())

print("\nUnités observées par variable :")
units_summary = (
    raw_long_df
    .groupby(["phenomenon", "sensor_unit"])
    .size()
    .reset_index(name="n_rows")
    .sort_values(["phenomenon", "n_rows"], ascending=[True, False])
)
display(units_summary)

download_log_df = pd.DataFrame(download_log)

print("\nRequêtes proches de la limite API :")
high_volume_sensors = download_log_df[
    download_log_df["n_rows"] >= 9000
].sort_values("n_rows", ascending=False)

if high_volume_sensors.empty:
    print("Aucune requête proche de la limite API.")
else:
    display(high_volume_sensors)

print("\nFICHIERS PRODUITS")
print("-" * 80)
print("1.", stations_metadata_path)
print("2.", raw_long_path)
print("3.", download_log_path)
print("4.", error_log_path)

print("\nAPERÇU DES VALEURS NUMÉRIQUES (colonne value)")
print("-" * 80)
display(
    raw_long_df.groupby("phenomenon")["value"]
    .agg(["count", "min", "max", "mean"])
    .reset_index()
)

print("\nÉtape suivante :")
print("  1bis (optionnel) — Notebook_1bis_Visualisation_lisible.ipynb")
print("       ou data/readable/opensensemap_apercu_2000.csv")
print("  2    — Notebook_2_Preparation_Controle_Qualite_openSenseMap.ipynb")
print("\nNotebook 1 terminé. Les données sont prêtes pour le Notebook 2.")
